# Endometrial Cancer (EC) Menopause-Timing PRS — All of Us CDR v9

Derives SBayesR weights for an age-at-natural-menopause polygenic score (Ruth et al. 2021), extracts and scores it for the final EC cohort from `EC_Cohort_Definition.ipynb`, and tests it as an additional covariate alongside the EC PRS (`EC_PRS_Analysis.ipynb`). Exploratory secondary analysis, not part of the primary EC PRS validation.

**Scope note (inputs and steps).** This notebook assumes the following are already prepared and available in the Workbench/bucket: the Ruth et al. 2021 age-at-natural-menopause GWAS summary-statistics file (`reprogen_ANM_201K_170621.txt` / `menopause_ANM.ma`), the UKB 10K HapMap3 LD reference panel (`ld_reference/band_ukb_10k_hm3/`), a compiled GCTB 2.5.5 executable, and `menopause_weights_hg38.csv` (the hg38-lifted SBayesR weights). The LD-reference load, the GCTB/SBayesR run, and the hg19->hg38 liftover of the derived weights were each run once only (commented out in the "SBayesR weight derivation" section now, not re-executed). The live, runnable steps in this notebook are: variant matching of the prepared weights against the extracted cohort genotypes, PLINK2 scoring via dsub, assembly and integrity-checking of the final per-person menopause PRS, and the menopause-PRS association analysis alongside the EC PRS.

## Setup

In [ ]:
import os
import subprocess
import pandas as pd
import numpy as np

project_id = os.environ.get('GOOGLE_PROJECT', '').strip().strip("'\"")
DSUB_USER_NAME = os.environ.get('OWNER_EMAIL', '').strip().strip("'\"").split('@')[0].replace('.', '-')
%run /home/jupyter/Var_set_2.ipynb
print(project_id, DSUB_USER_NAME)

## SBayesR weight derivation

In [ ]:
# Already run so commented out (LD reference load is slow)


# ld_dir = '/home/jupyter/ld_reference/band_ukb_10k_hm3'
# info_frames = []
# for chrom in range(1, 23):
#     path = f'{ld_dir}/band_chr{chrom}.ldm.sparse.info'
#     df = pd.read_csv(path, sep=r'\s+', usecols=['Chrom','ID','PhysPos','A1','A2'])
#     info_frames.append(df)
# ld_info = pd.concat(info_frames, ignore_index=True)
# ld_info['chrpos'] = 'chr' + ld_info['Chrom'].astype(str) + ':' + ld_info['PhysPos'].astype(str)
#
# reprogen = pd.read_csv('/home/jupyter/reprogen_ANM_201K_170621.txt', sep='\t')
# reprogen['chrpos'] = reprogen['SNP'].str.replace(':INDEL', '', regex=False)
# merged = reprogen.merge(ld_info[['chrpos','ID','A1','A2']], on='chrpos', how='inner')
# merged.to_csv('/home/jupyter/reprogen_ANM_with_rsid.txt', sep='\t', index=False)
# print(f"Matched to LD reference: {len(merged):,} / {len(reprogen):,}")


In [ ]:
# Already run so commented out. Long-running (~full session) and takes high memory to run


# gctb_bin = '/home/jupyter/gctb_extracted/gctb_2.5.5_Linux/gctb'
# mldm_file = '/home/jupyter/ld_reference/band_ukb_10k_hm3/ukb10k.mldm'
#
# cmd = f"""nohup {gctb_bin} --sbayes R \\
#     --mldm {mldm_file} \\
#     --pi 0.95,0.02,0.02,0.01 \\
#     --gamma 0.0,0.01,0.1,1 \\
#     --gwas-summary /home/jupyter/menopause_ANM.ma \\
#     --chain-length 10000 \\
#     --burn-in 2000 \\
#     --thin 10 \\
#     --out-freq 100 \\
#     --out /home/jupyter/menopause_sbayesr \\
#     > /home/jupyter/sbayesr_run.log 2>&1 &
# """
# subprocess.run(cmd, shell=True, capture_output=True, text=True)


In [ ]:
# Already run, commented out (row-by-row liftover over ~1.1M SNPs is slow; not worth rerunning unless menopause_weights_hg38.csv needs regenerating0
#
# from pyliftover import LiftOver
# lo = LiftOver('hg19', 'hg38')
# sbayesr = pd.read_csv('/home/jupyter/menopause_sbayesr.snpRes', sep=r'\s+')
#
# def lift_pos(row):
#     result = lo.convert_coordinate(f"chr{row['Chrom']}", int(row['Position']) - 1)
#     if result and len(result) > 0:
#         return result[0][1] + 1
#     return None
#
# sbayesr['position_hg38'] = sbayesr.apply(lift_pos, axis=1)
# sbayesr_valid = sbayesr[sbayesr['position_hg38'].notna()].copy()
# sbayesr_valid['position_hg38'] = sbayesr_valid['position_hg38'].astype(int)
# sbayesr_valid.to_csv('/home/jupyter/menopause_weights_hg38.csv', index=False)


In [ ]:
sbayesr_valid = pd.read_csv('/home/jupyter/menopause_weights_hg38.csv')
print(f"Total menopause-PRS SNPs: {len(sbayesr_valid):,}")
print(sbayesr_valid[['Name','Chrom','Position','position_hg38','A1','A2','A1Effect']].head())

## Variant matching and scoring

In [ ]:
missing = []
for chrom in range(1, 23):
    r = subprocess.run(
        ['gsutil', '-u', project_id, 'stat', f'{bucket_output}/menopause_extraction/chr{chrom}/prs_chr{chrom}_extracted.pgen'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        missing.append(chrom)
        print(f"chr{chrom}: MISSING")
    else:
        size_line = [l for l in r.stdout.splitlines() if 'Content-Length' in l]
        print(f"chr{chrom}: OK  {size_line[0].strip() if size_line else ''}")

print()
if missing:
    print(f"Missing chromosomes: {missing}")
else:
    print("All 22 menopause-extraction chromosomes present.")

In [ ]:
bed_ranges = sbayesr_valid[['Chrom', 'position_hg38']].copy()
bed_ranges['chrom_plink'] = 'chr' + bed_ranges['Chrom'].astype(str)
bed_ranges['start'] = bed_ranges['position_hg38'] - 1
bed_ranges['end'] = bed_ranges['position_hg38']
bed_ranges = bed_ranges[['chrom_plink', 'start', 'end']].drop_duplicates().sort_values(['chrom_plink', 'start'])
bed_ranges.to_csv('/home/jupyter/menopause_snp_positions.bed', sep='\t', header=False, index=False)
print(f"BED file: {len(bed_ranges):,} unique positions")

r_cohort = subprocess.run(['gsutil', '-u', project_id, 'cp', f'{bucket_output}/02_ec_final_cohort_excluded.csv', '/home/jupyter/02_ec_final_cohort_excluded.csv'], capture_output=True, text=True)
if r_cohort.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {r_cohort.returncode}): {r_cohort.stderr}")
final_cohort = pd.read_csv('/home/jupyter/02_ec_final_cohort_excluded.csv', low_memory=False)
print(f"Cohort: {len(final_cohort):,} ({(final_cohort['EC_case']==1).sum():,} cases)")

# Regenerating the BED file and --keep file from the live final_cohort loaded above (rather than reusing whatever keep-file already sits in the bucket)
# This way a rerun of the extraction/scoring cells always starts from a current, correct keep-file
keep_ids = final_cohort[['person_id']].copy()
keep_ids.columns = ['#IID']
keep_ids.to_csv('/home/jupyter/cohort_person_ids_menopause.txt', sep='\t', index=False)

for local_f, remote_name in [
    ('/home/jupyter/menopause_snp_positions.bed', 'menopause_snp_positions.bed'),
    ('/home/jupyter/cohort_person_ids_menopause.txt', 'cohort_person_ids_menopause.txt'),
]:
    r = subprocess.run(['gsutil', '-u', project_id, 'cp', local_f, f'{bucket_scripts}/{remote_name}'], capture_output=True, text=True)
    print(remote_name, '->', 'OK' if r.returncode == 0 else r.stderr)


In [ ]:
pvar_all = []
for chrom in range(1, 23):
    local_path = f'/home/jupyter/meno_chr{chrom}_extracted.pvar'
    r = subprocess.run(['gsutil', '-u', project_id, 'cp', f'{bucket_output}/menopause_extraction/chr{chrom}/prs_chr{chrom}_extracted.pvar', local_path], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"gsutil cp failed (exit {r.returncode}): {r.stderr}")

    with open(local_path) as f:
        header_line = None
        for i, line in enumerate(f):
            if not line.startswith('##'):
                header_line = i
                break

    df = pd.read_csv(local_path, sep='\t', skiprows=header_line, header=0)
    df.columns = ['CHROM', 'POS', 'ID', 'REF', 'ALT', 'INFO']
    df['chrom_num'] = chrom
    pvar_all.append(df[['chrom_num', 'POS', 'ID', 'REF', 'ALT']])
    print(f"chr{chrom}: {len(df):,} variants loaded")

pvar_combined = pd.concat(pvar_all, ignore_index=True)
print(f"\nTotal variants: {len(pvar_combined):,}")

In [ ]:
# Position-level duplication check before deduplication
# Doing this as keeping only the first .pvar record per chromosome:position can discard a valid allele match if multiple records exist there.
pvar_lookup_pre = pvar_combined.copy()
pvar_lookup_pre['plink_id_candidate'] = 'chr' + pvar_lookup_pre['chrom_num'].astype(str) + ':' + pvar_lookup_pre['POS'].astype(str)

position_counts = pvar_lookup_pre['plink_id_candidate'].value_counts()
multi_record_positions = position_counts[position_counts > 1]

print(f"Total extracted .pvar records: {len(pvar_lookup_pre):,}")
print(f"Unique chromosome:position combinations: {pvar_lookup_pre['plink_id_candidate'].nunique():,}")
print(f"Positions with more than one .pvar record: {len(multi_record_positions):,}")

weight_positions = set(sbayesr_valid['plink_id_candidate']) if 'plink_id_candidate' in sbayesr_valid.columns else \
    set('chr' + sbayesr_valid['Chrom'].astype(str) + ':' + sbayesr_valid['position_hg38'].astype(str))
multi_record_affecting_weights = weight_positions & set(multi_record_positions.index)
print(f"Weighted SNP positions affected by multi-record ambiguity: {len(multi_record_affecting_weights):,} "
      f"/ {len(weight_positions):,} ({100*len(multi_record_affecting_weights)/len(weight_positions):.3f}%)")

In [ ]:
sbayesr_valid['plink_id_candidate'] = 'chr' + sbayesr_valid['Chrom'].astype(str) + ':' + sbayesr_valid['position_hg38'].astype(str)

pvar_lookup = pvar_combined.copy()
pvar_lookup['plink_id_candidate'] = 'chr' + pvar_lookup['chrom_num'].astype(str) + ':' + pvar_lookup['POS'].astype(str)
pvar_lookup = pvar_lookup[['plink_id_candidate', 'ID', 'REF', 'ALT']].drop_duplicates(subset='plink_id_candidate')

merged = sbayesr_valid.merge(pvar_lookup, on='plink_id_candidate', how='left')
print(f"Weights matched to a real pvar position: {merged['REF'].notna().sum():,} / {len(merged):,}")

def determine_score_row(row):
    if pd.isna(row['REF']):
        return None, None, None
    alts = str(row['ALT']).split(',')
    if row['A1'] == row['REF'] and row['A2'] in alts:
        return row['ID'], row['A1'], row['A1Effect']
    elif row['A1'] in alts and row['A2'] == row['REF']:
        return row['ID'], row['A1'], row['A1Effect']
    else:
        return None, None, None

results = merged.apply(lambda r: pd.Series(determine_score_row(r)), axis=1)
results.columns = ['MATCHED_VARIANT_ID', 'SCORE_A1', 'SCORE_WEIGHT']
merged = pd.concat([merged.drop(columns=[c for c in ['MATCHED_VARIANT_ID','SCORE_A1','SCORE_WEIGHT'] if c in merged.columns]), results], axis=1)

n_resolved = merged['MATCHED_VARIANT_ID'].notna().sum()
print(f"Successfully resolved score entries: {n_resolved:,} / {len(merged):,}")

In [ ]:
score_file_menopause = merged[merged['MATCHED_VARIANT_ID'].notna()][['MATCHED_VARIANT_ID', 'SCORE_A1', 'SCORE_WEIGHT']].copy()
score_file_menopause.columns = ['ID', 'A1', 'WEIGHT']
print(f"Final menopause score file: {len(score_file_menopause):,} rows")

score_file_menopause.to_csv('/home/jupyter/menopause_score_file.txt', sep='\t', index=False)
r_meno_score = subprocess.run(['gsutil', '-u', project_id, 'cp', '/home/jupyter/menopause_score_file.txt', f'{bucket_scripts}/menopause_score_file.txt'], capture_output=True, text=True)
if r_meno_score.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {r_meno_score.returncode}): {r_meno_score.stderr}")

In [ ]:
score_script = '''#!/bin/bash
set -o pipefail
set -o errexit

chmod +x "${plink2_bin}"

mkdir -p /mnt/data/pfile
ln -s "${pgen_file}" /mnt/data/pfile/chr.pgen
ln -s "${psam_file}" /mnt/data/pfile/chr.psam
ln -s "${pvar_file}" /mnt/data/pfile/chr.pvar

"${plink2_bin}" \\
    --pfile /mnt/data/pfile/chr \\
    --score "${score_file}" 1 2 3 header no-mean-imputation list-variants \\
    --threads 7 \\
    --memory 30000 \\
    --out "${output_folder}/prs_chr${CHR}_score"

echo "Done chr${CHR}"
'''

with open('/home/jupyter/dsub_meno_score.sh', 'w') as f:
    f.write(score_script)
r_meno_script = subprocess.run(['gsutil', '-u', project_id, 'cp', '/home/jupyter/dsub_meno_score.sh', f'{bucket_scripts}/dsub_meno_score.sh'], capture_output=True, text=True)
if r_meno_script.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {r_meno_script.returncode}): {r_meno_script.stderr}")
print("Score script staged")

In [ ]:
# Submits 22 real dsub jobs against Google Batch (real compute cost and runtime)
# Uncomment to rerun from scratch only if menopause_extraction/ and menopause_scoring/ are confirmed stale relative to the current cohort (a cohort-coverage check is added later)


# def submit_chr_scoring_menopause(chrom):
#     output_folder = f"{bucket_output}/menopause_scoring/chr{chrom}"
#     logs_folder = f"{output_folder}/logs"
#     extracted_base = f"{bucket_output}/menopause_extraction/chr{chrom}/prs_chr{chrom}_extracted"

#     cmd = f"""dsub \\
#     --provider google-batch \\
#     --user-project {project_id} \\
#     --project {project_id} \\
#     --image "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod/ubuntu:latest" \\
#     --network "global/networks/network" \\
#     --use-private-address \\
#     --subnetwork "regions/us-central1/subnetworks/subnetwork" \\
#     --service-account "$(gcloud config get-value account)" \\
#     --user "{DSUB_USER_NAME}" \\
#     --regions us-central1 \\
#     --logging "{logs_folder}/{{job-id}}-{{task-id}}.log" \\
#     --boot-disk-size 30 \\
#     --disk-size 30 \\
#     --machine-type n2-standard-4 \\
#     --name "meno-sc-chr{chrom}" \\
#     --script "{bucket_scripts}/dsub_meno_score.sh" \\
#     --env GOOGLE_PROJECT={project_id} \\
#     --env output_folder="{output_folder}" \\
#     --env CHR={chrom} \\
#     --input plink2_bin="{bucket_scripts}/plink2" \\
#     --input pgen_file="{extracted_base}.pgen" \\
#     --input psam_file="{extracted_base}.psam" \\
#     --input pvar_file="{extracted_base}.pvar" \\
#     --input score_file="{bucket_scripts}/menopause_score_file.txt" \\
#     --output-recursive output_folder="{output_folder}"
# """
#     result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
#     print(result.stdout)
#     if result.returncode != 0:
#         print("ERROR:", result.stderr)
#     return result.stdout.strip().splitlines()[0] if result.stdout else None

# score_job_ids_menopause = {}
# for chrom in range(1, 23):
#     job_id = submit_chr_scoring_menopause(chrom)
#     score_job_ids_menopause[chrom] = job_id
#     print(f"chr{chrom}: {job_id}")

# print(f"\nSubmitted {len(score_job_ids_menopause)} menopause scoring jobs")

In [ ]:
# Checking job status (uncomment to rerun)


# result = subprocess.run(
#     ['dstat', '--provider', 'google-batch', '--project', project_id,
#      '--location', 'us-central1', '--jobs'] + list(score_job_ids_menopause.values()) +
#     ['--users', DSUB_USER_NAME, '--status', '*'],
#     capture_output=True, text=True
# )
# print(result.stdout)

In [ ]:
from functools import reduce

all_meno_scores = []
for chrom in range(1, 23):
    local_path = f'/home/jupyter/meno_chr{chrom}_score.sscore'
    r = subprocess.run(['gsutil', '-u', project_id, 'cp', f'{bucket_output}/menopause_scoring/chr{chrom}/prs_chr{chrom}_score.sscore', local_path], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"gsutil cp failed (exit {r.returncode}): {r.stderr}")
    df = pd.read_csv(local_path, sep='\t')
    df['chr_sum'] = df['SCORE1_AVG'] * df['ALLELE_CT']
    df = df[['#IID', 'chr_sum', 'ALLELE_CT']].rename(columns={'chr_sum': f'sum_chr{chrom}', 'ALLELE_CT': f'ct_chr{chrom}'})
    all_meno_scores.append(df)
    print(f"chr{chrom}: {len(df):,} people loaded")

meno_final_scores = reduce(lambda left, right: left.merge(right, on='#IID', how='outer'), all_meno_scores)
sum_cols = [c for c in meno_final_scores.columns if c.startswith('sum_chr')]
meno_final_scores['menopause_PRS_raw'] = meno_final_scores[sum_cols].sum(axis=1)

print(f"\nTotal people scored: {len(meno_final_scores):,}")
print(meno_final_scores['menopause_PRS_raw'].describe())

In [ ]:
r_current = subprocess.run(['gsutil', '-u', project_id, 'cp', f'{bucket_output}/02_ec_final_cohort_excluded.csv', '/home/jupyter/02_ec_final_cohort_excluded.csv'], capture_output=True, text=True)
if r_current.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {r_current.returncode}): {r_current.stderr}")
current_cohort = pd.read_csv('/home/jupyter/02_ec_final_cohort_excluded.csv', low_memory=False)
print(f"Current cohort: {len(current_cohort):,}")

meno_ana = meno_final_scores.merge(
    current_cohort[['person_id', 'EC_case']].rename(columns={'person_id':'#IID'}),
    on='#IID', how='inner'
)
print(f"Merged with current cohort: {len(meno_ana):,}")
print(f"Cases: {int((meno_ana['EC_case']==1).sum()):,}  Controls: {int((meno_ana['EC_case']==0).sum()):,}")

ct_cols = [c for c in meno_ana.columns if c.startswith('ct_chr')]
n_incomplete = meno_ana[ct_cols].isna().any(axis=1).sum()
print(f"\nPeople with at least one missing chromosome allele count: {n_incomplete}")
if n_incomplete == 0:
    print("OK: every person has real data on all 22 chromosomes.")

# Safeguard for if a larger/more-complete menopause_scores_final.csv already exists on disk
# Use that instead of a smaller fresh merge (guards against reloading a stale intermediate)
# The actual correctness gate is the integrity checks in the next cell
_saved_path = '/home/jupyter/menopause_scores_final.csv'
if os.path.exists(_saved_path):
    _saved = pd.read_csv(_saved_path, low_memory=False)
    _saved_n = len(_saved)
    if _saved_n > len(meno_ana):
        print(f"\nNOTE: menopause_scores_final.csv on disk already covers more people "
              f"({_saved_n:,}) than this fresh merge ({len(meno_ana):,}) -- using the on-disk file.")
        meno_ana = _saved
    elif _saved_n == len(meno_ana) and len(current_cohort) > len(meno_ana):
        print(f"\nWARNING: both this fresh merge and the on-disk file ({_saved_n:,}) are "
              f"smaller than the current full cohort ({len(current_cohort):,}). A fresh "
              f"extraction/scoring run is needed.")

if len(meno_ana) < len(current_cohort):
    print(f"\nWARNING: meno_ana ({len(meno_ana):,}) is still smaller than the current "
          f"cohort ({len(current_cohort):,}). Re-run the extraction/scoring cells from scratch.")
else:
    print(f"\nConfirmed: meno_ana covers the full current cohort (N={len(meno_ana):,}).")


In [ ]:
# Pre-save integrity checks against the live current_cohort, regardless of whether meno_ana came from a fresh merge or the on-disk recovery above.
assert meno_ana['#IID'].is_unique, "Duplicate #IID in meno_ana"
assert current_cohort['person_id'].is_unique, "Duplicate person_id in current_cohort"
assert set(meno_ana['#IID']) == set(current_cohort['person_id']), (
    "meno_ana's participant set does not exactly match the current cohort -- "
    "it is not safe to save this as menopause_scores_final.csv"
)

labels_check = meno_ana[['#IID', 'EC_case']].merge(
    current_cohort[['person_id', 'EC_case']],
    left_on='#IID', right_on='person_id',
    validate='one_to_one', suffixes=('_prs', '_cohort')
)
assert (labels_check['EC_case_prs'] == labels_check['EC_case_cohort']).all(), (
    "EC_case labels in meno_ana disagree with the current cohort's labels"
)

sum_cols_check = [f'sum_chr{i}' for i in range(1, 23)]
ct_cols_check = [f'ct_chr{i}' for i in range(1, 23)]
assert meno_ana[sum_cols_check + ct_cols_check].notna().all().all(), (
    "meno_ana has missing per-chromosome sum/count values"
)
assert meno_ana[ct_cols_check].gt(0).all().all(), (
    "meno_ana has a zero or negative allele count for at least one chromosome"
)
np.testing.assert_allclose(
    meno_ana['menopause_PRS_raw'], meno_ana[sum_cols_check].sum(axis=1), rtol=1e-7, atol=1e-7
)
print("Menopause PRS pre-save integrity checks passed.")

# Secondary signal (row-count regression) which is less complete, the integrity checks above are the actual correctness gate
_saved_path = '/home/jupyter/menopause_scores_final.csv'
if os.path.exists(_saved_path):
    _existing_n = sum(1 for _ in open(_saved_path)) - 1
    if len(meno_ana) < _existing_n:
        print(f"\nNOTE: meno_ana ({len(meno_ana):,} rows) has fewer rows than the file "
              f"already on disk ({_existing_n:,}). The integrity checks above passed, so "
              f"this is being saved anyway -- but if unexpected, stop and investigate first.")

ctrl_m = meno_ana.loc[meno_ana['EC_case']==0, 'menopause_PRS_raw'].mean()
ctrl_s = meno_ana.loc[meno_ana['EC_case']==0, 'menopause_PRS_raw'].std()
meno_ana['menopause_PRS_z'] = (meno_ana['menopause_PRS_raw'] - ctrl_m) / ctrl_s

print("menopause_PRS_z by case status:")
print(meno_ana.groupby('EC_case')['menopause_PRS_z'].describe())

meno_ana.to_csv('/home/jupyter/menopause_scores_final.csv', index=False)
r = subprocess.run(['gsutil','-u',project_id,'cp','/home/jupyter/menopause_scores_final.csv',
    f'{bucket_output}/menopause_scores_final.csv'], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {r.returncode}): {r.stderr}")
print(r.stdout, r.stderr)
print(f"\nSaved: N={len(meno_ana):,}")


## Menopause-timing PRS association

In [ ]:
r_baseline = subprocess.run(['gsutil','-u',project_id,'cp', f'{bucket_output}/03_ec_cohort_with_baseline_chars.csv', '/home/jupyter/cohort_baseline_meno.csv'], capture_output=True, text=True)
if r_baseline.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {r_baseline.returncode}): {r_baseline.stderr}")
cohort_bmi = pd.read_csv('/home/jupyter/cohort_baseline_meno.csv', low_memory=False)

r_ec_prs = subprocess.run(['gsutil','-u',project_id,'cp', f'{bucket_output}/prs_scores_final_complete.csv', '/home/jupyter/ec_prs_for_meno_merge.csv'], capture_output=True, text=True)
if r_ec_prs.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {r_ec_prs.returncode}): {r_ec_prs.stderr}")
ec_prs = pd.read_csv('/home/jupyter/ec_prs_for_meno_merge.csv', low_memory=False)

merged_all = ec_prs[['#IID', 'PRS_z', 'EC_case']].merge(
    meno_ana[['#IID', 'menopause_PRS_z']], on='#IID', how='inner'
).merge(
    cohort_bmi[['person_id', 'bmi_baseline', 'age_display']].rename(columns={'person_id':'#IID'}),
    on='#IID', how='left'
).merge(
    ec_prs[[c for c in ec_prs.columns if c.startswith('PC')][:10] + ['#IID']],
    on='#IID', how='left'
)
print(f"Merged: {len(merged_all):,}")
print(f"Missing menopause_PRS_z: {merged_all['menopause_PRS_z'].isna().sum()}")
print(f"Missing BMI: {merged_all['bmi_baseline'].isna().sum()}")
print(f"Missing age: {merged_all['age_display'].isna().sum()}")

# Cohort-coverage check here as the inner join above silently drops anyone in ec_prs but missing from meno_ana 
# This is scored by a separate pipeline, possibly on an earlier cohort snapshot
# Warns only as re-running the menopause extraction/scoring is a real multi-hour cost, so this never triggers that automatically.
missing_from_meno = set(ec_prs['#IID']) - set(meno_ana['#IID'])
meno_cohort_is_complete = (len(missing_from_meno) == 0) and (len(merged_all) == len(ec_prs))

if not meno_cohort_is_complete:
    print("\nWARNING: merged_all does not cover the full current EC PRS cohort.")
    print(f"  {len(missing_from_meno):,} cohort member(s) are in the current EC PRS cohort "
          f"(ec_prs, N={len(ec_prs):,}) but missing from the menopause PRS scores (meno_ana, "
          f"N={len(meno_ana):,}) -- merged_all ends up smaller than the true cohort as a result "
          f"(N={len(merged_all):,}).")
    print("  Likely means the menopause extraction predates a later cohort change. To fix,")
    print("  regenerate the --keep list from the current final_cohort and re-run the menopause")
    print("  extraction/scoring from scratch. Not triggered automatically here since it costs")
    print("  real time/compute -- confirm the gap is real before deciding to fix it.")
else:
    print(f"\nConfirmed: merged_all covers the full current EC PRS cohort "
          f"(N={len(merged_all):,}, matching ec_prs's N={len(ec_prs):,}).")


In [ ]:
import statsmodels.api as sm
import numpy as np
from sklearn.metrics import roc_auc_score

pc_cols = sorted([c for c in merged_all.columns if c.startswith('PC') and c[2:].isdigit()], key=lambda x: int(x[2:]))[:10]

ctrl = merged_all[merged_all['EC_case']==0]
age_m, age_s = ctrl['age_display'].mean(), ctrl['age_display'].std()
bmi_m, bmi_s = ctrl['bmi_baseline'].mean(), ctrl['bmi_baseline'].std()
merged_all['age_z'] = (merged_all['age_display'] - age_m) / age_s
merged_all['bmi_z'] = (merged_all['bmi_baseline'] - bmi_m) / bmi_s

# Model 3 (baseline for comparison): PRS + age + BMI + PCs
d3 = merged_all.dropna(subset=['PRS_z', 'age_z', 'bmi_z'] + pc_cols)
X3 = sm.add_constant(d3[['PRS_z', 'age_z', 'bmi_z'] + pc_cols])
y3 = d3['EC_case']
m3 = sm.Logit(y3, X3).fit(disp=0)
auc3 = roc_auc_score(y3, m3.predict(X3))

# Model 4 (new): PRS + age + BMI + menopause_PRS_z + PCs
d4 = merged_all.dropna(subset=['PRS_z', 'age_z', 'bmi_z', 'menopause_PRS_z'] + pc_cols)
X4 = sm.add_constant(d4[['PRS_z', 'age_z', 'bmi_z', 'menopause_PRS_z'] + pc_cols])
y4 = d4['EC_case']
m4 = sm.Logit(y4, X4).fit(disp=0)
auc4 = roc_auc_score(y4, m4.predict(X4))

def report(m, X, y, auc, label):
    print(f"\n=== {label} (N={len(y):,}, cases={int(y.sum()):,}) ===")
    for var in [c for c in X.columns if c != 'const' and not c.startswith('PC')]:
        orr = np.exp(m.params[var]); ci = np.exp(m.conf_int().loc[var])
        print(f"  {var:<18} OR {orr:.4f} (95% CI {ci[0]:.4f}-{ci[1]:.4f}), p={m.pvalues[var]:.2e}")
    print(f"  AUC: {auc:.4f}")

report(m3, X3, y3, auc3, "Model 3: PRS + age + BMI + 10 PCs")
report(m4, X4, y4, auc4, "Model 4: PRS + age + BMI + menopause_PRS_z + 10 PCs")
print(f"\nAUC: {auc3:.4f} -> {auc4:.4f}")

In [ ]:
# Confirm d3 (EC PRS full model) and d4 (EC PRS + menopause PRS) use the same participantcset
# This way, the menopause PRS's marginal contribution is a paired comparison.
ids_d3 = set(d3['#IID']) if '#IID' in d3.columns else set(d3.index)
ids_d4 = set(d4['#IID']) if '#IID' in d4.columns else set(d4.index)
only_in_d3 = ids_d3 - ids_d4
if ids_d3 != ids_d4:
    print(f"NOTE: d3 and d4 do NOT use identical participant sets. d3-only: {len(only_in_d3):,}, "
          f"d4-only: {len(ids_d4 - ids_d3):,}. This is expected if menopause_PRS_z has additional")
    print(f"missingness beyond PRS_z/age_z/bmi_z -- the comparison between m3 and m4 should be")
    print(f"interpreted with this in mind, or re-run both models on the ids_d3 & ids_d4 intersection")
    print(f"for a strictly paired comparison.")
else:
    print(f"Confirmed: d3 and d4 use identical participant sets, N={len(ids_d3):,}.")

In [ ]:
# Sanity check: does menopause_PRS_z show ANY association with EC_case on its own before adding other covariates?
d_uni = merged_all.dropna(subset=['menopause_PRS_z'] + pc_cols)
X_uni = sm.add_constant(d_uni[['menopause_PRS_z'] + pc_cols])
y_uni = d_uni['EC_case']
m_uni = sm.Logit(y_uni, X_uni).fit(disp=0)

orr = np.exp(m_uni.params['menopause_PRS_z'])
ci = np.exp(m_uni.conf_int().loc['menopause_PRS_z'])
print(f"menopause_PRS_z alone (+ PCs, no age/BMI): OR {orr:.4f} (95% CI {ci[0]:.4f}-{ci[1]:.4f}), p={m_uni.pvalues['menopause_PRS_z']:.2e}")


In [ ]:
participant_score_corr = merged_all['PRS_z'].corr(merged_all['menopause_PRS_z'])
print(f"Correlation between participants' EC PRS_z and menopause_PRS_z: {participant_score_corr:.4f}")
print(f"N = {merged_all[['PRS_z','menopause_PRS_z']].dropna().shape[0]:,}")

ctrl_corr = merged_all.loc[merged_all['EC_case']==0, ['PRS_z','menopause_PRS_z']].dropna()
print(f"\nControls only: r = {ctrl_corr['PRS_z'].corr(ctrl_corr['menopause_PRS_z']):.4f} (N={len(ctrl_corr):,})")

In [ ]:
# SNP-level and weight-level overlap between the EC and menopause score files -- 
# This checks how related the two scores actually are, alongside the person-level PRS_z correlation above.
subprocess.run(['gsutil','-u',project_id,'cp', f'{bucket_scripts}/prs_score_file_v2.txt', '/home/jupyter/ec_score_check.txt'], capture_output=True, text=True, check=True)
subprocess.run(['gsutil','-u',project_id,'cp', f'{bucket_scripts}/menopause_score_file.txt', '/home/jupyter/meno_score_check.txt'], capture_output=True, text=True, check=True)

ec_score = pd.read_csv('/home/jupyter/ec_score_check.txt', sep='\t')
meno_score = pd.read_csv('/home/jupyter/meno_score_check.txt', sep='\t')

print(f"EC PRS SNPs: {len(ec_score):,}")
print(f"Menopause PRS SNPs: {len(meno_score):,}")

ec_ids = set(ec_score['ID'])
meno_ids = set(meno_score['ID'])
snp_overlap = ec_ids & meno_ids

print(f"\nOverlapping SNP IDs: {len(snp_overlap):,}")
print(f"  As % of EC PRS SNPs: {100*len(snp_overlap)/len(ec_ids):.2f}%")
print(f"  As % of menopause PRS SNPs: {100*len(snp_overlap)/len(meno_ids):.2f}%")

# High positional overlap with near-zero weight correlation would mean the two scores draw on the same SNPs but encode largely unrelated effects. 
# Comparing raw WEIGHT values is only meaningful if both files score the same allele (A1) at each SNP, checked below.
ec_overlap = ec_score[ec_score['ID'].isin(snp_overlap)][['ID','WEIGHT']].rename(columns={'WEIGHT':'EC_WEIGHT'})
meno_overlap = meno_score[meno_score['ID'].isin(snp_overlap)][['ID','WEIGHT']].rename(columns={'WEIGHT':'MENO_WEIGHT'})

weight_compare = ec_overlap.merge(meno_overlap, on='ID')
print(f"\nSNPs compared: {len(weight_compare):,}")

weight_corr = weight_compare['EC_WEIGHT'].corr(weight_compare['MENO_WEIGHT'])
print(f"Correlation between EC PRS weights and menopause PRS weights (at shared SNPs, "
      f"allele orientation not yet checked): {weight_corr:.4f}")

# Allele-orientation check of do both files count the same effect allele (A1) at each shared SNP? 
# If not, the weight correlation above could be misleading in either direction.
shared = ec_score[['ID', 'A1', 'WEIGHT']].merge(
    meno_score[['ID', 'A1', 'WEIGHT']],
    on='ID', suffixes=('_ec', '_meno'), validate='one_to_one'
)
same_allele = shared['A1_ec'].eq(shared['A1_meno'])
print(f"\nShared SNPs: {len(shared):,}")
print(f"Same scored allele (A1) in both files: {same_allele.mean():.2%}")
print(f"Weight correlation where scored alleles agree: "
      f"{shared.loc[same_allele, 'WEIGHT_ec'].corr(shared.loc[same_allele, 'WEIGHT_meno']):.4f}")
if (~same_allele).any():
    print(f"Weight correlation where scored alleles DISAGREE ({(~same_allele).sum():,} SNPs, "
          f"sign-flipped before comparing): "
          f"{(-shared.loc[~same_allele, 'WEIGHT_ec']).corr(shared.loc[~same_allele, 'WEIGHT_meno']):.4f}")

# Harmonising all shared SNPs to the menopause-scored allele (flipping the EC weight's sign wherever the two files scored the opposite allele)
# Computing one overall correlation -- the two subgroup correlations above can't simply be averaged, since they can have different means/variances.
shared['EC_WEIGHT_aligned'] = np.where(
    shared['A1_ec'] == shared['A1_meno'],
    shared['WEIGHT_ec'],
    -shared['WEIGHT_ec']
)
r_harmonised = shared['EC_WEIGHT_aligned'].corr(shared['WEIGHT_meno'])
print(f"\nOverall allele-harmonised weight correlation (all {len(shared):,} shared SNPs): {r_harmonised:.4f}")
